# BroilerVision — YOLO26x Fine-Tuning on Labeled Broiler Barn Dataset
Run this in Google Colab (Runtime → Change runtime type → GPU, T4 or A100).

Dataset: 50 hand-labeled frames from the barn overhead clip, exported from Roboflow via curl link.

Small dataset (50 frames) — heavy augmentation + a longer patience window matters more here than epoch count.

In [ ]:
# 1. Confirm GPU
!nvidia-smi

In [ ]:
# 2. Install deps (upgrade to latest ultralytics for YOLO26 support)
!pip install -U ultralytics -q

In [ ]:
# 3. Download dataset from Roboflow (curl export link)
!mkdir -p /content/dataset
%cd /content/dataset
!curl -L "https://app.roboflow.com/ds/5BxBAPbIeF?key=RTnW5zNYvH" > roboflow.zip
!unzip -q roboflow.zip
!rm roboflow.zip
!cat data.yaml

In [ ]:
# 4. Train YOLO26x on the dataset
# imgsz=1920 matches the inference resolution the demo pipeline uses —
# that's what it took to catch far-background birds in the fisheye frame,
# so training at the same scale keeps train/inference consistent.
# Drop to imgsz=1280 and/or batch=4 if you hit OOM on a T4.
from ultralytics import YOLO

model = YOLO('yolo26x.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=150,
    imgsz=1920,
    batch=8,
    name='broilervision_yolo26x',
    device=0,
    patience=30,       # small dataset — give it room before early-stopping
    augment=True,
    mosaic=1.0,
    degrees=10.0,       # mild rotation aug — fisheye barn cam, birds at all angles
    fliplr=0.5,
    hsv_v=0.3,          # lighting varies a lot barn-to-barn (dim post-transfer, bright daylight)
)
print('Best weights:', results.save_dir)

In [ ]:
# 5. Validate and check mAP
model_best = YOLO(str(results.save_dir) + '/weights/best.pt')
val = model_best.val(data='/content/dataset/data.yaml')
print(f'mAP50: {val.box.map50:.3f}')
print(f'mAP50-95: {val.box.map:.3f}')

In [ ]:
# 6. Sanity-check predictions on a training frame before downloading
import glob
sample = glob.glob('/content/dataset/train/images/*')[0]
model_best.predict(sample, conf=0.15, imgsz=1920, save=True)
print('Saved annotated sample to runs/detect/predict/')

In [ ]:
# 7. Download best.pt to your machine
from google.colab import files
files.download(str(results.save_dir) + '/weights/best.pt')
# Then put it in C:\Users\leodo\claude\broilervision\broilervision_best.pt
# and tell me — I'll swap it into process_footage.py in place of yolo26x.pt.
#
# IMPORTANT: with this fine-tuned single-class model, class 0 = chicken
# (not COCO class 14 = bird). process_footage.py's BIRD_CLASS constant
# needs to change from 14 to 0 when you switch models.